Ejemplo de uso de **BBDD** **SQLite** desde **Python**

In [24]:
import sqlite3
import pandas as pd

In [25]:
# CONEXIÓN Y CONFIGURACIÓN INICIAL
conexion = sqlite3.connect('cartera_financiera.db')
cursor = conexion.cursor()

# Activar la validación de Claves Foréneas
cursor.execute("PRAGMA foreign_keys = ON;")

In [26]:
# Limpiamos todo para comenzar la creación de tablas desde cero si ejecutamos varias veces
cursor.execute("DROP TABLE IF EXISTS Prestamos;")
cursor.execute("DROP TABLE IF EXISTS Clientes;")

# CREACIÓN DE TABLAS, especificando nombres de columnas y sus tipos
cursor.execute('''
    CREATE TABLE Clientes (
      ID_Cliente INTEGER PRIMARY KEY,
      Empresa TEXT,
      Sector TEXT,
      Score_Crediticio FLOAT
    )
''')

cursor.execute('''
CREATE TABLE Prestamos (
    ID_Operacion TEXT PRIMARY KEY,
    ID_Cliente INTEGER,
    Importe DECIMAL(10,2),
    FOREIGN KEY (ID_Cliente) REFERENCES Clientes(ID_Cliente)
)
''')

In [27]:
# 3. INYECCIÓN DE DATOS FINANCIEROS
datos_clientes = [
    (101, 'TechNova SL', 'Tecnología', 0.85),
    (102, 'RetailSur SA', 'Comercio', 0.42),
    (103, 'BioFarma', 'Salud', 0.91),
    (104, 'Construcciones Iberia', 'Inmobiliario', 0.65),
    (105, 'AgroSur', 'Agricultura', 0.55),
    (106, 'Fintech Partners', 'Finanzas', 0.88)
]

datos_prestamos = [
    ('A-001', 101, 50000),
    ('A-002', 101, 15000),
    ('B-001', 102, 120000),
    ('B-002', 102, 35000),
    ('C-001', 103, 300000),
    ('C-002', 103, 50000),
    ('D-001', 104, 75000),
    ('E-001', 105, 12000),
    ('F-001', 106, 90000)
]

# Insertar la información masivamente
cursor.executemany("INSERT INTO Clientes VALUES (?, ?, ?, ?)", datos_clientes)
cursor.executemany("INSERT INTO Prestamos VALUES (?, ?, ?)", datos_prestamos)

# Guardar cambios
conexion.commit()
print("Base de datos 'cartera_financiera.db' generada")



Base de datos 'cartera_financiera.db' generada


In [28]:
print("Extrayendo de dataset con features para ML (INNER JOIN) ---")

query_ml = """
SELECT C.Empresa, C.Sector, C.Score_Crediticio, P.Importe
FROM Clientes C
INNER JOIN Prestamos P ON C.ID_Cliente = P.ID_Cliente
WHERE P.Importe > 40000;
"""

df_modelo = pd.read_sql_query(query_ml, conexion)
df_modelo

Extrayendo de dataset con features para ML (INNER JOIN) ---


,Empresa,Sector,Score_Crediticio,Importe
0,TechNova SL,Tecnología,0.85,50000
1,RetailSur SA,Comercio,0.42,120000
2,BioFarma,Salud,0.91,300000
3,BioFarma,Salud,0.91,50000
4,Construcciones Iberia,Inmobiliario,0.65,75000
5,Fintech Partners,Finanzas,0.88,90000


In [29]:
print("---MEJORA: Extrayendo la Feature Matrix Agregada para ML ---")

# Combinamos JOIN, WHERE y GROUP BY en la "Query Maestra"
query_ml_avanzada = """
SELECT
    C.Empresa,
    C.Sector,
    C.Score_Crediticio,
    SUM(P.Importe) as Riesgo_Total_Operaciones_Grandes,
    COUNT(P.ID_Operacion) as Num_Operaciones
FROM Clientes C
INNER JOIN Prestamos P ON C.ID_Cliente = P.ID_Cliente
WHERE P.Importe > 40000
GROUP BY C.ID_Cliente, C.Empresa, C.Sector, C.Score_Crediticio;
"""

# Volcamos a Pandas
df_modelo_agregado = pd.read_sql_query(query_ml_avanzada, conexion)
print(df_modelo_agregado)

# Siempre cerramos la conexión al terminar
conexion.close()

---MEJORA: Extrayendo la Feature Matrix Agregada para ML ---
                 Empresa        Sector  Score_Crediticio  \
0            TechNova SL    Tecnología              0.85   
1           RetailSur SA      Comercio              0.42   
2               BioFarma         Salud              0.91   
3  Construcciones Iberia  Inmobiliario              0.65   
4       Fintech Partners      Finanzas              0.88   

   Riesgo_Total_Operaciones_Grandes  Num_Operaciones  
0                             50000                1  
1                            120000                1  
2                            350000                2  
3                             75000                1  
4                             90000                1  
